In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

!pip install \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    peft \
    rouge-score \
    evaluate \
    open_clip_torch \
    scikit-learn \
    nltk

!pip install faiss-cpu
!pip install torchxrayvision

In [32]:
# Imports all the necessary libraries
import os
import gc
import json
import time
import random
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional

import faiss
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import densenet121
import torchxrayvision as xrv

from PIL import Image

from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

import re
from sklearn.metrics import f1_score

from peft import (

    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

from transformers.models.minicpmv4_6.modeling_minicpmv4_6 import (
    MiniCPMV4_6ForConditionalGeneration
)

warnings.filterwarnings("ignore")

import nltk

nltk.download("punkt")

[nltk_data] Downloading package punkt to C:\Users\INSTRUMENT
[nltk_data]     LAB\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [7]:
DATA_ROOT = Path("C:/Mimic/mimic-data/data/data")

VECTOR_DIR = DATA_ROOT / "vector_store"
DATA_DIR = DATA_ROOT
IMAGE_ROOT = Path("C:/Mimic/mimics/test_dataset/images")

VIEW_CLASSIFIER_PATH = (
    DATA_ROOT /
    "checkpoints" /
    "view_classifier.pt"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Detected device: {DEVICE}")

ACTIVE_MODEL = "minicpm-v-4.6"
MAX_TEST_SAMPLES = 40
TOP_K = 5

AP_WEIGHT = 0.65
LV_WEIGHT = 0.35

Detected device: cuda


In [8]:
# ============================================================
# CONFIGURATION
# ============================================================

VLM_MODELS = {
    "qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct",
    "nv-reason-cxr-3b": "nvidia/NV-Reason-CXR-3B",
    "minicpm-v-4.6": "openbmb/MiniCPM-V-4.6",
}

In [9]:
def cleanup_cuda():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def cosine_similarity(a, b):

    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)

    return np.dot(a, b)

In [10]:
# Load all embeddings and meta data
print("Loading metadata and embeddings...")

# ============================================================
# METADATA
# ============================================================

with open(DATA_DIR / "parsed_reports.json") as f:
    parsed_reports = json.load(f)

with open(DATA_DIR / "temporal_patients.json") as f:
    temporal_patients = json.load(f)

with open(DATA_DIR / "trajectories.json") as f:
    trajectories = json.load(f)

with open(DATA_DIR / "temporal_windows.json") as f:
    temporal_windows = json.load(f)

with open(DATA_DIR / "view_index.json") as f:
    view_index = json.load(f)

# ============================================================
# EMBEDDINGS
# ============================================================

report_embeddings = np.load(
    VECTOR_DIR / "reports_embeddings.npy"
)

trajectory_embeddings = np.load(
    VECTOR_DIR / "trajectories_embeddings.npy"
)

window_embeddings = np.load(
    VECTOR_DIR / "windows_embeddings.npy"
)

img_ap_embeddings = np.load(
    VECTOR_DIR / "img_ap_embeddings.npy"
)

img_lv_embeddings = np.load(
    VECTOR_DIR / "img_lv_embeddings.npy"
)

fused_embeddings = np.load(
    VECTOR_DIR / "fused_reports_embeddings.npy"
)

print("Metadata + embeddings loaded")

Loading metadata and embeddings...
Metadata + embeddings loaded


In [11]:
with open(VECTOR_DIR / "reports_meta.json") as f:

    reports_meta = json.load(f)

report_keys = reports_meta["keys"]

with open(VECTOR_DIR / "trajectories_meta.json") as f:

    trajectories_meta = json.load(f)

trajectory_keys = trajectories_meta["keys"]

with open(
    VECTOR_DIR / "trajectories_metadata.pkl",
    "rb"
) as f:

    trajectory_metadata = pickle.load(f)

with open(VECTOR_DIR / "windows_meta.json") as f:

    windows_meta = json.load(f)

window_keys = windows_meta["keys"]

with open(
    VECTOR_DIR / "windows_metadata.pkl",
    "rb"
) as f:

    window_metadata = pickle.load(f)

with open(VECTOR_DIR / "img_ap_meta.json") as f:

    ap_meta = json.load(f)

ap_keys = ap_meta["keys"]


with open(VECTOR_DIR / "img_lv_meta.json") as f:

    lv_meta = json.load(f)

lv_keys = lv_meta["keys"]

In [12]:
# Build All FAISS Indices
def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    normalized = np.ascontiguousarray(
        embeddings.astype(np.float32)
    )
    
    faiss.normalize_L2(normalized)
    index.add(normalized)
    return index


report_index = build_index(report_embeddings)
trajectory_index = build_index(trajectory_embeddings)
window_index = build_index(window_embeddings)
ap_index = build_index(img_ap_embeddings)
lv_index = build_index(img_lv_embeddings)
fused_index = build_index(fused_embeddings)

In [13]:
# Load BiomedCLIP + Query Encoding + Temporal Intent Detection
import open_clip

print("Loading BiomedCLIP...")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)

clip_tokenizer = open_clip.get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)

clip_model = clip_model.to(DEVICE)
clip_model.eval()

# ============================================================
# QUERY ENCODERS
# ============================================================

def encode_text_query(text):

    with torch.no_grad():

        tokens = clip_tokenizer([text]).to(DEVICE)

        emb = clip_model.encode_text(tokens)

        emb = F.normalize(emb, dim=-1)

    return emb.cpu().numpy()[0]


def encode_image_query(image_path):

    image = clip_preprocess(
        Image.open(image_path).convert("RGB")
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        emb = clip_model.encode_image(image)

        emb = F.normalize(emb, dim=-1)

    return emb.cpu().numpy()[0]

# ============================================================
# TEMPORAL INTENT DETECTION
# ============================================================

def detect_temporal_intent(text):

    text = text.lower()

    worsening_words = [
        "worsening",
        "progression",
        "increase",
        "larger",
        "worse",
    ]

    improving_words = [
        "improving",
        "resolved",
        "better",
        "decrease",
        "reduced",
    ]

    stable_words = [
        "stable",
        "unchanged",
        "persistent",
    ]

    if any(w in text for w in worsening_words):
        return "worsening"

    if any(w in text for w in improving_words):
        return "improving"

    if any(w in text for w in stable_words):
        return "stable"

    return "unknown"

Loading BiomedCLIP...


In [14]:
# ============================================================
# VIEW CLASSIFIER
# ============================================================

class ViewClassifierModel(nn.Module):

    def __init__(self):

        super().__init__()

        # ====================================================
        # BACKBONE
        # ====================================================

        self.backbone = xrv.models.DenseNet(
            weights="densenet121-res224-mimic_ch"
        )

        # ====================================================
        # CLASSIFIER HEAD
        # ====================================================

        self.classifier = nn.Linear(
            1024,
            2
        )

    def forward(self, x):

        # ====================================================
        # FEATURE EXTRACTION
        # ====================================================

        feats = self.backbone.features(x)

        feats = F.relu(feats, inplace=True)

        feats = F.adaptive_avg_pool2d(
            feats,
            (1, 1)
        )

        feats = torch.flatten(
            feats,
            1
        )

        # ====================================================
        # CLASSIFICATION
        # ====================================================

        logits = self.classifier(feats)

        return logits


print("Loading view classifier...")

view_classifier = ViewClassifierModel()

checkpoint = torch.load(
    VIEW_CLASSIFIER_PATH,
    map_location=DEVICE,
)

# ============================================================
# HANDLE DIFFERENT CHECKPOINT FORMATS
# ============================================================

if "state_dict" in checkpoint:

    checkpoint = checkpoint["state_dict"]

view_classifier.load_state_dict(
    checkpoint,
    strict=False,
)

view_classifier = view_classifier.to(DEVICE)

view_classifier.eval()

print("View classifier loaded")

Loading view classifier...
View classifier loaded


In [15]:
view_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [16]:
def predict_view(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    image = view_transform(image)

    image = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        logits = view_classifier(image)

        pred = torch.argmax(
            logits,
            dim=1
        ).item()

    # 0 = AP/PA
    # 1 = LATERAL

    if pred == 0:
        return "AP"

    return "LATERAL"

In [17]:
@dataclass
class RetrievalResult:

    reports: List[Dict]

    fused_reports: List[Dict]

    windows: List[Dict]

    trajectories: List[Dict]

In [18]:
# ============================================================
# ProcessedQuery Structure
# ============================================================

@dataclass
class ProcessedQuery:

    report_embedding: np.ndarray

    fused_query: np.ndarray

    window_query: np.ndarray

    trajectory_query: np.ndarray

    temporal_status: str

    pathology_labels: List[str]

In [19]:
# Temporal Reranking
def rerank_temporal_alignment(
    temporal_intent,
    retrieved,
):

    reranked = []

    for item in retrieved:

        score = item["score"]

        trend = item.get("trend", "unknown")

        if trend == temporal_intent:
            score += 0.15

        reranked.append((score, item))

    reranked = sorted(
        reranked,
        key=lambda x: x[0],
        reverse=True,
    )

    return [x[1] for x in reranked]

In [20]:
# ============================================================
# Lightweight QueryProcessor
# ============================================================

class QueryProcessor:

    def __init__(self):

        self.pathology_keywords = {

            "edema": [
                "edema",
                "fluid overload",
            ],

            "effusion": [
                "pleural effusion",
                "effusion",
            ],

            "pneumonia": [
                "pneumonia",
                "consolidation",
                "opacity",
            ],

            "pneumothorax": [
                "pneumothorax",
            ],

            "cardiomegaly": [
                "cardiomegaly",
                "enlarged cardiac silhouette",
            ],
        }

    # ========================================================
    # PATHOLOGY EXTRACTION
    # ========================================================

    def extract_pathologies(
        self,
        text,
    ):

        text = text.lower()

        labels = []

        for pathology, keywords in self.pathology_keywords.items():

            if any(k in text for k in keywords):

                labels.append(pathology)

        return labels

    # ========================================================
    # TEMPORAL FEATURES
    # ========================================================

    def build_window_features(
        self,
        temporal_status,
    ):

        trend = 0.0

        if temporal_status == "improving":
            trend = -1.0

        elif temporal_status == "worsening":
            trend = 1.0

        intensity = abs(trend)

        return np.array(
            [trend, intensity],
            dtype=np.float32,
        )

    def build_trajectory_features(
        self,
        temporal_status,
        labels,
    ):

        improving = 0.0
        worsening = 0.0

        if temporal_status == "improving":
            improving = 1.0

        elif temporal_status == "worsening":
            worsening = 1.0

        severity = min(
            len(labels) / 5.0,
            1.0,
        )

        return np.array([
            improving,
            worsening,
            severity,
            float("edema" in labels),
            float("effusion" in labels),
            float("pneumonia" in labels),
            float("cardiomegaly" in labels),
            float("pneumothorax" in labels),
        ], dtype=np.float32)

    # ========================================================
    # MAIN PROCESS
    # ========================================================

    def process(
        self,
        findings_text,
        image_paths=None,
    ):

        # ====================================================
        # TEXT EMBEDDING
        # ====================================================

        text_emb = encode_text_query(
            findings_text
        )

        # ====================================================
        # IMAGE EMBEDDING
        # ====================================================

        image_emb = None

        if image_paths:

            img_embs = []

            for p in image_paths:

                emb = encode_image_query(p)

                img_embs.append(emb)

            image_emb = np.mean(
                img_embs,
                axis=0,
            )

        # ====================================================
        # FUSED QUERY
        # ====================================================

        fused_query = text_emb.copy()

        if image_emb is not None:

            fused_query = (
                0.7 * text_emb +
                0.3 * image_emb
            )

        fused_query = fused_query.astype(
            np.float32
        )

        # ====================================================
        # TEMPORAL STATUS
        # ====================================================

        temporal_status = detect_temporal_intent(
            findings_text
        )

        # ====================================================
        # PATHOLOGY LABELS
        # ====================================================

        pathology_labels = self.extract_pathologies(
            findings_text
        )

        # ====================================================
        # WINDOW QUERY
        # ====================================================

        window_features = self.build_window_features(
            temporal_status
        )

        window_query = np.concatenate([
            fused_query,
            window_features,
        ])

        # ====================================================
        # TRAJECTORY QUERY
        # ====================================================

        trajectory_features = self.build_trajectory_features(
            temporal_status,
            pathology_labels,
        )

        trajectory_query = np.concatenate([
            fused_query,
            trajectory_features,
        ]).astype(np.float32)

        return ProcessedQuery(

            report_embedding=text_emb,

            fused_query=fused_query,

            window_query=window_query,

            trajectory_query=trajectory_query,

            temporal_status=temporal_status,

            pathology_labels=pathology_labels,
        )

In [21]:
class TemporalRetriever:

    def __init__(self):
        pass

    def retrieve(
        self,
        query,
        top_k=5,
    ):

        temporal_intent = query.temporal_status

        report_query = query.report_embedding

        fused_query = query.fused_query
        
        window_query = query.window_query
        
        trajectory_query = query.trajectory_query

        report_query = report_query.reshape(1, -1)
        fused_query = fused_query.reshape(1, -1)
        window_query = window_query.reshape(1, -1)
        trajectory_query = trajectory_query.reshape(1, -1)
        
        faiss.normalize_L2(report_query)
        faiss.normalize_L2(fused_query)
        faiss.normalize_L2(window_query)
        faiss.normalize_L2(trajectory_query)

        # ==========================================
        # REPORT RETRIEVAL
        # ==========================================

        scores, indices = report_index.search(
            report_query,
            top_k,
        )

        reports = []

        for score, idx in zip(scores[0], indices[0]):

            study_key = report_keys[idx]

            report = parsed_reports.get(
                study_key,
                {}
            )

            reports.append({
                "study_key": study_key,
                "score": float(score),
                "findings": report.get("findings", ""),
                "impression": report.get("impression", ""),
            })

        # ==========================================
        # FUSED RETRIEVAL
        # ==========================================
        
        scores, indices = fused_index.search(
            fused_query,
            top_k,
        )
        
        fused_reports = []
        
        for score, idx in zip(scores[0], indices[0]):
        
            study_key = report_keys[idx]
        
            report = parsed_reports.get(
                study_key,
                {}
            )
        
            fused_reports.append({
                "study_key": study_key,
                "score": float(score),
                "findings": report.get("findings", ""),
                "impression": report.get("impression", ""),
            })

        # ==========================================
        # WINDOW RETRIEVAL
        # ==========================================

        scores, indices = window_index.search(
            window_query,
            top_k,
        )

        windows = []

        for score, idx in zip(scores[0], indices[0]):

            window_key = window_keys[idx]
            
            meta = window_metadata.get(
                window_key,
                {}
            )

            windows.append({
                "window_key": meta.get(
                    "window_key",
                    window_key
                ),
                "score": float(score),
                "trend": meta.get("trend", "unknown"),
                "text": meta.get("text", ""),
            })

        windows = rerank_temporal_alignment(
            temporal_intent,
            windows,
        )

        # ==========================================
        # TRAJECTORY RETRIEVAL
        # ==========================================

        scores, indices = trajectory_index.search(
            trajectory_query,
            top_k,
        )

        trajs = []

        for score, idx in zip(scores[0], indices[0]):

            traj_key = trajectory_keys[idx]
            
            meta = trajectory_metadata.get(
                traj_key,
                {}
            )

            trajs.append({
                "patient_id": meta.get(
                    "patient_id",
                    traj_key
                ),
                "score": float(score),
                "trend": meta.get("trend", "unknown"),
                "summary": meta.get("summary", ""),
            })

        trajs = rerank_temporal_alignment(
            temporal_intent,
            trajs,
        )

        return RetrievalResult(
            reports=reports,
            fused_reports=fused_reports,
            windows=windows,
            trajectories=trajs,
        )

In [22]:
# ============================================================
# TEMPORAL MULTIMODAL RAG PROMPT
# ============================================================

def build_prompt(

    patient_timeline_text,
    retrieval_result,
):

    # ========================================================
    # RETRIEVED REPORTS
    # ========================================================

    reports_text = "\n".join([

        r["findings"][:200]

        for r in retrieval_result.reports[:3]
    ])


    # ========================================================
    # MULTIMODAL RETRIEVAL
    # ========================================================

    fused_text = "\n".join([

        r["findings"][:200]

        for r in retrieval_result.fused_reports[:3]
    ])


    # ========================================================
    # TEMPORAL WINDOWS
    # ========================================================

    windows_text = "\n".join([

        w["text"][:200]

        for w in retrieval_result.windows[:3]
    ])


    # ========================================================
    # TRAJECTORIES
    # ========================================================

    trajectory_text = "\n".join([

        t["summary"][:200]

        for t in retrieval_result.trajectories[:3]
    ])


    # ========================================================
    # FINAL PROMPT
    # ========================================================

    prompt = f"""
You are an expert thoracic radiologist AI specialized in
longitudinal chest X-ray interpretation.

PATIENT TIMELINE CONTEXT:
{patient_timeline_text}

RETRIEVED PRIOR REPORTS:
{reports_text}

RETRIEVED MULTIMODAL CASES:
{fused_text}

TEMPORAL PROGRESSION MEMORY:
{windows_text}

LONGITUDINAL TRAJECTORY MEMORY:
{trajectory_text}

TASK:
Generate a clinically grounded chest X-ray report
for the MOST RECENT study using temporal reasoning
across the provided timeline and retrieved memories.

Focus on:
- interval progression
- worsening vs improvement
- persistent findings
- disease evolution
- temporal consistency

Output ONLY:

FINDINGS:
...

IMPRESSION:
...

Rules:
- Use concise radiology-report style
- No bullet points
- No conversational text
- No explanations
- Keep longitudinal reasoning clinically grounded
"""

    return prompt.strip()


def retrieval_baseline(retrieval_result):

    findings = []
    impressions = []

    for r in retrieval_result.reports[:3]:

        findings.append(r.get("findings", ""))
        impressions.append(r.get("impression", ""))

    findings = " ".join(findings)
    impressions = " ".join(impressions)

    return f"""
FINDINGS:
{findings}

IMPRESSION:
{impressions}
"""

In [23]:
# ============================================================
# Evaluation Metrics
# ============================================================

PATHOLOGY_TERMS = {
    "Atelectasis": ["atelectasis", "collapse"],
    "Cardiomegaly": ["cardiomegaly", "heart enlarged"],
    "Consolidation": ["consolidation"],
    "Edema": ["edema", "congestion"],
    "Enlarged Cardiomediastinum": ["mediastinum"],
    "Fracture": ["fracture"],
    "Lung Lesion": ["nodule", "mass", "lesion"],
    "Lung Opacity": ["opacity", "infiltrate", "ground glass"],
    "No Finding": ["normal", "clear"],
    "Pleural Effusion": ["effusion"],
    "Pleural Other": ["pleural thickening"],
    "Pneumonia": ["pneumonia"],
    "Pneumothorax": ["pneumothorax"],
    "Support Devices": ["tube", "line", "catheter", "device"],
}


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


# ============================================================
# NEGATION-AWARE PATHOLOGY EXTRACTION
# ============================================================

NEGATION_WORDS = [
    "no",
    "without",
    "absent",
    "negative for",
]


def extract_labels(text):

    text = clean_text(text)

    labels = {}


    # ========================================================
    # EACH PATHOLOGY GROUP
    # ========================================================

    for pathology, keywords in PATHOLOGY_TERMS.items():

        found = False

        negated = False


        # ====================================================
        # KEYWORD SEARCH
        # ====================================================

        for kw in keywords:

            matches = list(
                re.finditer(
                    re.escape(kw.lower()),
                    text
                )
            )

            if not matches:
                continue


            found = True


            # =================================================
            # NEGATION WINDOW
            # =================================================

            idx = matches[-1].start()

            window = text[
                max(0, idx - 40): idx
            ]

            negated = any(

                neg_word in window

                for neg_word in NEGATION_WORDS
            )

            if negated:
                break


        # ====================================================
        # STORE LABEL
        # ====================================================

        if found:

            labels[pathology] = (

                "negative"

                if negated

                else "positive"
            )

    return labels


# ============================================================
# BLEU
# ============================================================

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction,
)

smoothie = SmoothingFunction().method4


def compute_bleu(
    pred,
    gt,
):

    pred_tokens = clean_text(pred).split()

    gt_tokens = [
        clean_text(gt).split()
    ]

    try:

        score = sentence_bleu(

            gt_tokens,

            pred_tokens,

            smoothing_function=smoothie,
        )

    except:

        score = 0.0

    return score


# ============================================================
# ROUGE-L
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)

def compute_rouge(
    pred,
    gt,
):

    try:

        score = rouge.score(
            gt,
            pred,
        )["rougeL"].fmeasure

    except:

        score = 0.0

    return score


# ============================================================
# CHEXPERT F1
# ============================================================

def compute_chexpert_f1(
    pred,
    gt,
):

    pred_labels = extract_labels(pred)

    gt_labels = extract_labels(gt)

    all_keys = set(
        pred_labels.keys()
    ) | set(
        gt_labels.keys()
    )

    if len(all_keys) == 0:

        return 1.0

    tp = 0
    fp = 0
    fn = 0

    for key in all_keys:

        pred_val = pred_labels.get(key)
        gt_val = gt_labels.get(key)

        if pred_val == gt_val and pred_val is not None:

            tp += 1

        elif pred_val is not None and gt_val is None:

            fp += 1

        elif pred_val != gt_val:

            fp += 1
            fn += 1

    precision = tp / (tp + fp + 1e-8)

    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-8)
    )

    return f1

# ─────────────────────────────────────────────
# TEMPORAL CHANGE PATTERNS
# ─────────────────────────────────────────────

IMPROVE = [r"improv", r"resolv", r"decreas", r"reduced", r"less", r"clearing"]
WORSEN  = [r"worsen", r"increas", r"progress", r"larger", r"worse"]
STABLE  = [r"unchanged", r"stable", r"no\s+change", r"similar", r"persistent"]
NEW     = [r"new", r"developed", r"appeared", r"interval\s+development"]
RESOLVED= [r"resolved", r"no\s+longer", r"disappeared", r"cleared"]

# ============================================================
# TEMPORAL PATTERN GROUPS
# ============================================================

TEMPORAL_PATTERNS = {

    "improved": IMPROVE,

    "worsened": WORSEN,

    "stable": STABLE,

    "new": NEW,

    "resolved": RESOLVED,
}


# ============================================================
# TEMPORAL STATE EXTRACTION
# ============================================================

def extract_temporal_states(text):

    text = clean_text(text)

    states = set()

    for state, patterns in TEMPORAL_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                states.add(state)

                break

    if len(states) == 0:

        states.add("unknown")

    return states


# ============================================================
# TEMPORAL AGREEMENT
# ============================================================

def compute_temporal_agreement(

    pred,

    gt,
):

    pred_states = extract_temporal_states(pred)

    gt_states = extract_temporal_states(gt)

    if "unknown" in gt_states:

        return 1.0

    intersection = len(
        pred_states & gt_states
    )

    union = len(
        pred_states | gt_states
    )

    return intersection / (union + 1e-8)


# ============================================================
# PATHOLOGY-SPECIFIC TEMPORAL EXTRACTION
# ============================================================

def extract_pathology_progression(text):

    text = clean_text(text)

    progression = {}


    # ========================================================
    # EACH PATHOLOGY
    # ========================================================

    for pathology, keywords in PATHOLOGY_TERMS.items():

        pathology_found = False


        # ====================================================
        # PATHOLOGY DETECTION
        # ====================================================

        for kw in keywords:

            if kw.lower() in text:

                pathology_found = True
                break


        if not pathology_found:
            continue


        # ====================================================
        # TEMPORAL STATE DETECTION
        # ====================================================

        detected_state = "present"


        for state, patterns in TEMPORAL_PATTERNS.items():

            for pattern in patterns:

                if re.search(pattern, text):

                    detected_state = state
                    break

            if detected_state != "present":
                break


        progression[pathology] = detected_state


    return progression


def _pathology_keys(text):

    return set(extract_pathology_progression(text).keys())


def compute_pathology_progression_accuracy(pred, gt):

    pred_keys = _pathology_keys(pred)
    gt_keys = _pathology_keys(gt)

    if len(pred_keys) == 0 and len(gt_keys) == 0:
        return 1.0

    if len(pred_keys) == 0 or len(gt_keys) == 0:
        return 0.0

    overlap = len(pred_keys & gt_keys)
    union = len(pred_keys | gt_keys)

    return overlap / (union + 1e-8)
    


# ============================================================
# LONGITUDINAL ENTITY CONSISTENCY
# ============================================================

def compute_longitudinal_consistency(

    generated_reports,
):

    if len(generated_reports) < 2:

        return 1.0

    consistency_scores = []

    prev_labels = extract_labels(
        generated_reports[0]
    )

    for report in generated_reports[1:]:

        curr_labels = extract_labels(report)

        overlap = len(

            set(prev_labels.items()) &
            set(curr_labels.items())
        )

        total = len(

            set(prev_labels.items()) |
            set(curr_labels.items())
        )

        if total == 0:

            consistency_scores.append(1.0)

        else:

            consistency_scores.append(
                overlap / total
            )

        prev_labels = curr_labels

    return np.mean(consistency_scores)


# ============================================================
# RETRIEVAL PRECISION@K
# ============================================================

def compute_retrieval_precision(

    retrieved_reports,

    gt_text,
):

    gt_keys = _pathology_keys(gt_text)

    if len(gt_keys) == 0:

        return 1.0

    scores = []

    for r in retrieved_reports:

        retrieved_keys = _pathology_keys(
            r.get("findings", "")
        )

        if len(retrieved_keys) == 0:
            continue

        overlap = len(retrieved_keys & gt_keys)

        precision = overlap / len(gt_keys)

        scores.append(precision)

    if len(scores) == 0:

        return 0.0

    return np.mean(scores)


# ============================================================
# RETRIEVAL HIT@K
# ============================================================

def compute_retrieval_hit(

    retrieved_reports,

    gt_text,
):

    gt_keys = _pathology_keys(gt_text)

    if len(gt_keys) == 0:

        return 1.0

    for r in retrieved_reports:

        retrieved_keys = _pathology_keys(
            r.get("findings", "")
        )

        if len(retrieved_keys & gt_keys) > 0:

            return 1.0

    return 0.0

In [24]:
# VLM Backend Base
class VLMBackend:

    def generate(
        self,
        prompt,
        image_paths=None,
    ):
        raise NotImplementedError

In [25]:
class QwenBackend(VLMBackend):
    
    def __init__(
        self,
        model_name,
    ):

        cleanup_cuda()

        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is not available in this notebook kernel. Use a CUDA-enabled PyTorch environment and restart the kernel."
            )

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )

        self.processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        self.model = (
            Qwen2_5_VLForConditionalGeneration
            .from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                quantization_config=quant_config,
                device_map={"": 0},
                trust_remote_code=True,
            )
        )

        # ========================================================
        # PREPARE MODEL FOR QLORA
        # ========================================================
        
        self.model = prepare_model_for_kbit_training(
            self.model
        )

        # ========================================================
        # LORA CONFIG
        # ========================================================
        
        lora_config = LoraConfig(
        
            r=16,
        
            lora_alpha=32,
        
            target_modules=[
        
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
            ],
        
            lora_dropout=0.05,
        
            bias="none",
        
            task_type="CAUSAL_LM",
        )

        # ========================================================
        # APPLY LORA
        # ========================================================
        
        self.model = get_peft_model(
        
            self.model,
        
            lora_config,
        )
        self.model.config.use_cache = False
        self.model.gradient_checkpointing_enable()
        self.model.print_trainable_parameters()

    def generate(
        self,
        prompt,
        image_paths=None,
    ):
    
        content = []
    
        # ========================================================
        # IMAGES
        # ========================================================
    
        if image_paths:
    
            for img in image_paths:
    
                content.append({
                    "type": "image",
                    "image": img,
                })
    
        # ========================================================
        # TEXT
        # ========================================================
    
        content.append({
            "type": "text",
            "text": prompt,
        })
    
        messages = [
            {
                "role": "user",
                "content": content,
            }
        ]
    
        # ========================================================
        # CHAT TEMPLATE
        # ========================================================
    
        text_prompt = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    
        # ========================================================
        # PROCESS INPUTS
        # ========================================================
    
        inputs = self.processor(
            text=[text_prompt],
            images=[
                Image.open(p).convert("RGB").resize((224, 224))
                for p in image_paths
            ] if image_paths else None,
            return_tensors="pt",
            max_pixels=224*224,
        )
    
        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }
    
        # ========================================================
        # GENERATION
        # ========================================================
    
        with torch.no_grad():
    
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
            )
    
        # ========================================================
        # REMOVE PROMPT TOKENS
        # ========================================================
    
        generated_ids = outputs[
            :,
            inputs["input_ids"].shape[1]:
        ]
    
        # ========================================================
        # DECODE
        # ========================================================
    
        generated = self.processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )[0]

        del outputs
        del generated_ids
        del inputs
        
        torch.cuda.empty_cache()
        gc.collect()
    
        return generated.strip()

In [26]:
class NVReasonCXRBackend:

    def __init__(
        self,
        model_name,
    ):

        cleanup_cuda()

        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is not available in this notebook kernel. Use a CUDA-enabled PyTorch environment and restart the kernel."
            )

        # ====================================================
        # QUANTIZATION CONFIG
        # ====================================================

        bnb_config = BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_compute_dtype=torch.float16,

            bnb_4bit_quant_type="nf4",

            bnb_4bit_use_double_quant=True,
        )

        # ====================================================
        # PROCESSOR
        # ====================================================

        self.processor = AutoProcessor.from_pretrained(

            model_name,

            trust_remote_code=True,
        )

        # ====================================================
        # BASE MODEL
        # ====================================================

        self.model = (
            Qwen2_5_VLForConditionalGeneration
            .from_pretrained(

                model_name,

                torch_dtype=torch.float16,

                device_map={"": 0},

                quantization_config=bnb_config,

                trust_remote_code=True,

                attn_implementation="sdpa",
            )
        )

        # ========================================================
        # PREPARE MODEL FOR QLORA
        # ========================================================
        
        self.model = prepare_model_for_kbit_training(
            self.model
        )

        # ====================================================
        # LoRA CONFIG
        # ====================================================

        lora_config = LoraConfig(

            r=8,

            lora_alpha=16,

            lora_dropout=0.05,

            bias="none",

            task_type="CAUSAL_LM",

            target_modules=[

                "q_proj",

                "k_proj",

                "v_proj",

                "o_proj",
            ],
        )

        # ====================================================
        # APPLY PEFT / LoRA
        # ====================================================

        self.model = get_peft_model(

            self.model,

            lora_config,
        )

        # ====================================================
        # TRAINING SETTINGS
        # ====================================================

        self.model.config.use_cache = False

        self.model.gradient_checkpointing_enable()

        self.model.print_trainable_parameters()

        self.model.train()

    # ========================================================
    # GENERATION
    # ========================================================

    def generate(
        self,
        prompt,
        image_paths=None,
    ):

        content = []

        # ====================================================
        # IMAGES
        # ====================================================

        images = []

        if image_paths:

            for p in image_paths[:2]:

                try:

                    img = (
                        Image.open(p)
                        .convert("RGB")
                        .resize((224, 224))
                    )

                    images.append(img)

                    content.append({

                        "type": "image",

                        "image": p,
                    })

                except Exception as e:

                    print(f"Image loading failed: {p}")

        # ====================================================
        # TEXT
        # ====================================================

        content.append({

            "type": "text",

            "text": prompt,
        })

        messages = [

            {
                "role": "user",

                "content": content,
            }
        ]

        # ====================================================
        # CHAT TEMPLATE
        # ====================================================

        text_prompt = self.processor.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True,
        )

        # ====================================================
        # PROCESS INPUTS
        # ====================================================

        inputs = self.processor(

            text=[text_prompt],

            images=images if len(images) > 0 else None,

            padding=True,

            truncation=True,

            max_length=512,

            max_pixels=224 * 224,

            return_tensors="pt",
        )

        # ====================================================
        # MOVE TO CUDA
        # ====================================================

        inputs = {

            k: v.to(DEVICE)

            for k, v in inputs.items()
        }

        # ====================================================
        # GENERATION
        # ====================================================

        with torch.no_grad():

            outputs = self.model.generate(

                **inputs,

                max_new_tokens=256,

                do_sample=False,
            )

        # ====================================================
        # REMOVE PROMPT TOKENS
        # ====================================================

        generated_ids = outputs[
            :,
            inputs["input_ids"].shape[1]:
        ]

        # ====================================================
        # DECODE
        # ====================================================

        generated = self.processor.batch_decode(

            generated_ids,

            skip_special_tokens=True,
        )[0]

        # ====================================================
        # CLEANUP
        # ====================================================

        del outputs
        del generated_ids
        del inputs

        torch.cuda.empty_cache()

        return generated.strip()

In [27]:
class MiniCPMBackend(VLMBackend):

    def __init__(self, model_name):

        cleanup_cuda()

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        self.processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        self.model = (
            MiniCPMV4_6ForConditionalGeneration
            .from_pretrained(
                model_name,
                quantization_config=bnb_config,
                torch_dtype=torch.float16,
                trust_remote_code=True,
                device_map="auto",
            )
        )

        # ========================================================
        # PREPARE MODEL FOR QLORA
        # ========================================================
        
        self.model = prepare_model_for_kbit_training(
            self.model
        )

        # ====================================================
        # LoRA CONFIG
        # ====================================================

        lora_config = LoraConfig(

            r=8,

            lora_alpha=16,

            lora_dropout=0.05,

            bias="none",

            task_type="CAUSAL_LM",

            target_modules=[

                "q_proj",

                "k_proj",

                "v_proj",

                "o_proj",
            ],
        )

        # ====================================================
        # APPLY PEFT / LoRA
        # ====================================================

        self.model = get_peft_model(

            self.model,

            lora_config,
        )

        # ====================================================
        # TRAINING SETTINGS
        # ====================================================

        self.model.config.use_cache = False

        self.model.gradient_checkpointing_enable()

        self.model.print_trainable_parameters()

        self.model.train()

    # ========================================================
    # GENERATION
    # ========================================================

    def generate(
        self,
        prompt,
        image_paths=None,
    ):

        content = []

        # ====================================================
        # IMAGES
        # ====================================================

        if image_paths:

            for img in image_paths[:2]:

                content.append({

                    "type": "image",

                    "image": img,
                })

        # ====================================================
        # TEXT
        # ====================================================

        content.append({

            "type": "text",

            "text": prompt,
        })

        messages = [

            {
                "role": "user",

                "content": content,
            }
        ]

        # ====================================================
        # CHAT TEMPLATE
        # ====================================================

        text_prompt = self.processor.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True,
        )

        # ====================================================
        # LOAD IMAGES
        # ====================================================

        images = [

            Image.open(p)
            .convert("RGB")
            .resize((128,128))

            for p in image_paths[:2]

        ] if image_paths else None

        # ====================================================
        # PROCESS INPUTS
        # ====================================================

        inputs = self.processor(

            text=[text_prompt],

            images=images,

            return_tensors="pt",
        )

        model_device = next(
            self.model.parameters()
        ).device

        inputs = {

            k: v.to(model_device)

            for k, v in inputs.items()
        }

        # ====================================================
        # GENERATE
        # ====================================================

        with torch.inference_mode():

            outputs = self.model.generate(

                **inputs,

                max_new_tokens=256,

                do_sample=False,
            )

        # ====================================================
        # REMOVE PROMPT
        # ====================================================

        generated_ids = outputs[
            :,
            inputs["input_ids"].shape[1]:
        ]

        # ====================================================
        # DECODE
        # ====================================================

        generated = self.processor.batch_decode(

            generated_ids,

            skip_special_tokens=True,
        )[0]

        del outputs
        del generated_ids
        del inputs
        
        torch.cuda.empty_cache()
        gc.collect()
    
        return generated.strip()


In [29]:
# ============================================================
# BUILD TRAINING SAMPLES (SELF-CONTAINED)
# ============================================================

retriever = TemporalRetriever()
query_processor = QueryProcessor()
training_samples = []

TEST_REPORT_DIR = Path(
    "C:/Mimic/mimics/test_dataset/reports"
)

TEST_IMAGE_DIR = Path(
    "C:/Mimic/mimics/test_dataset/images"
)

# ============================================================
# LOAD MIMIC METADATA
# ============================================================

metadata_df = pd.read_csv(

    "C:\Mimic\mimic-data\mimic-cxr-2.0.0-metadata.csv"
)

# ============================================================
# CREATE TRUE STUDY DATETIME
# ============================================================

metadata_df["StudyDate"] = (
    metadata_df["StudyDate"]
    .astype(str)
)

metadata_df["StudyTime"] = (

    metadata_df["StudyTime"]

    .fillna(0)

    .astype(int)

    .astype(str)

    .str.zfill(6)
)

metadata_df["study_datetime"] = pd.to_datetime(

    metadata_df["StudyDate"]
    +
    metadata_df["StudyTime"],

    format="%Y%m%d%H%M%S",

    errors="coerce",
)

# ============================================================
# STUDY DATETIME LOOKUP
# ============================================================

study_datetime_lookup = {

    f"s{row.study_id}": row.study_datetime

    for _, row in metadata_df.iterrows()
}

patient_trajectories = {}

print("Indexing image files for training...")
image_lookup = {}
for ext in ["jpg", "jpeg", "png"]:
    for img_path in TEST_IMAGE_DIR.rglob(f"*.{ext}"):
        study_key = img_path.parent.name
        image_lookup.setdefault(study_key, []).append(str(img_path))

print("Building patient trajectories for training...")
for report_path in TEST_REPORT_DIR.rglob("*.txt"):
    try:
        patient_id = report_path.parts[-3]
        study_key = report_path.stem

        with open(report_path) as f:
            report_text = f.read().strip()

        if not report_text:
            continue

        sample = {
            "study_key": study_key,
            "findings": report_text,
            "ground_truth": report_text,
            "image_paths": image_lookup.get(study_key, []),
        }

        patient_trajectories.setdefault(patient_id, []).append(sample)
    except Exception as e:
        print(f"Skipping {report_path}: {e}")

for patient_id in patient_trajectories:
    patient_trajectories[patient_id] = sorted(
        patient_trajectories[patient_id],
        key=lambda x:
        study_datetime_lookup.get(
            x["study_key"],
            pd.Timestamp.min
        ),
    )

patient_ids = sorted(patient_trajectories.keys())
print(f"Total patients available: {len(patient_ids)}")

MAX_TRAIN_PATIENTS = 250
selected_patients = patient_ids[:MAX_TRAIN_PATIENTS]

print(f"Creating training samples from {len(selected_patients)} patients...")
for patient_id in selected_patients:
    trajectory = patient_trajectories[patient_id]

    for idx, item in enumerate(trajectory):
        history = trajectory[:idx]
        history_text = "\n".join([h["findings"][:250] for h in history[-3:]])

        if history_text.strip():
            temporal_query = f"""
Previous studies:
{history_text}

Current study:
{item['findings']}
""".strip()
        else:
            temporal_query = f"Current study:\n{item['findings']}"

        processed_query = query_processor.process(
            findings_text=temporal_query,
            image_paths=item["image_paths"],
        )

        retrieval = retriever.retrieve(
            processed_query,
            top_k=TOP_K,
        )

        prompt = build_prompt(
            patient_timeline_text=temporal_query,
            retrieval_result=retrieval,
        )

        training_samples.append({
            "prompt": prompt,
            "answer": item["ground_truth"],
            "image_path": item["image_paths"],
            "study_datetime": study_datetime_lookup.get(study_key),
        })

print(f"Training samples prepared: {len(training_samples)}")

Indexing image files for training...
Building patient trajectories for training...
Total patients available: 525
Creating training samples from 250 patients...
Training samples prepared: 4562


In [ ]:
# ============================================================
# MODEL + TOKENIZER (FOR LoRA TRAINING)
# ============================================================
from tqdm.auto import tqdm

if "model" not in globals():
    if ACTIVE_MODEL == "qwen2.5-vl-3b":
        model = QwenBackend(VLM_MODELS["qwen2.5-vl-3b"])
    elif ACTIVE_MODEL == "nv-reason-cxr-3b":
        model = NVReasonCXRBackend(VLM_MODELS["nv-reason-cxr-3b"])
    elif ACTIVE_MODEL == "minicpm-v-4.6":
        model = MiniCPMBackend(VLM_MODELS["minicpm-v-4.6"])
    else:
        raise ValueError(f"Unknown ACTIVE_MODEL: {ACTIVE_MODEL}")


# ============================================================
# DATA COLLATOR
# ============================================================

def collate_fn(samples):

    print(f"\nProcessing batch of {len(samples)} samples")

    texts = []

    all_images = []

    for idx, sample in enumerate(samples):

        print(f"Loading sample {idx+1}")

        start = time.time()

        images = [

            Image.open(img_path)
            .convert("RGB")
            .resize((128,128))

            for img_path in sample["image_path"][:2]
        ]

        print(
            "Images loaded in:",
            round(time.time()-start,2),
            "sec"
        )

        content = []

        for _ in images:

            content.append({
                "type":"image"
            })

        content.append({

            "type":"text",

            "text": sample["prompt"][:300]
        })

        messages = [

            {
                "role":"user",
                "content":content,
            },

            {
                "role":"assistant",
                "content":[
                    {
                        "type":"text",
                        "text":sample["answer"][:300],
                    }
                ]
            }
        ]

        print("Applying template...")

        text = model.processor.apply_chat_template(

            messages,

            tokenize=False,
        )

        texts.append(text)

        all_images.append(images)

    print("Running processor...")

    start = time.time()

    batch = model.processor(

        text=texts,

        images=all_images,

        padding=True,

        truncation=True,

        max_length=256,

        return_tensors="pt",
    )

    print(
        "Processor completed in:",
        round(time.time()-start,2),
        "sec"
    )

    labels = batch["input_ids"].clone()

    labels[
        labels ==
        model.processor.tokenizer.pad_token_id
    ] = -100

    batch["labels"] = labels

    print("Batch ready")

    return batch

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(
    training_samples
)

torch.cuda.empty_cache()
gc.collect()

11041

In [ ]:
OUTPUT_ROOT = Path("C:\Mimic")
training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / f"{ACTIVE_MODEL}_temporal_lora"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=2,
    logging_steps=1,
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
)

torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
# Trainer
model.model.train()
trainer = Trainer(
    model=model.model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn,
)

start = time.time()
batch = collate_fn([dataset[0]])

print(
    "TOTAL BATCH TIME:",
    round(time.time()-start,2),
    "sec"
)

trainer.train()


Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.97 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready
TOTAL BATCH TIME: 0.97 sec

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.27 sec
Applying template...
Running processor...
Processor completed in: 0.01 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.18 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.22 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.53 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.17 sec
Applying template...
Running processor...
Processor completed in: 0.01 sec
Batch ready


Step,Training Loss
1,3.627791



Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.13 sec
Applying template...
Running processor...
Processor completed in: 0.01 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.35 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.61 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.37 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.38 sec
Applying template...
Running processor...
Processor completed in: 0.0 sec
Batch ready

Processing batch of 1 samples
Loading sample 1
Images loaded in: 0.19 sec
Applying template...
Running processor...
Processor completed in: 0.01 sec
Batch ready

Processing batch of 1 samples
L

KeyboardInterrupt: 

In [ ]:
model.processor.save_pretrained(
    str(OUTPUT_ROOT / f"{ACTIVE_MODEL}_temporal_lora")
)

model.model.save_pretrained(
    str(OUTPUT_ROOT / f"{ACTIVE_MODEL}_temporal_lora")
)